In [ ]:
from text_to_sql import run_text_to_sql
from models.openaiapipoint import llm
from get_schema import getschema_agent
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
import matplotlib.pyplot as plt
import pandas as pd
import re


# ---------------------------
# PROMPT
# ---------------------------
PROMPT = """
You analyze user requests that may involve data retrieval, SQL generation, or chart creation.

Extract and output ONLY valid JSON with these fields:

- sql_intent: A natural-language description of the data to retrieve.
  No SQL keywords. No query structure.
  Only describe the dataset, grouping, metrics, and filters required.
  Example: "Return total revenue grouped by month for the current year."

- chartprompt: If the user requests any visualization 
  (pie, bar, line, histogram, scatter, area, etc.),
  produce a clean, minimal instruction suitable for PandasAI SmartDataframe.
  The instruction must:
    * Clearly state the chart type.
    * Name the column(s) to plot.
    * Include a required figure size instruction like:
        "Set figure size to 10 by 6."
    * Use matplotlib.
    * Use ASCII text only.
    * Contain no SQL and no dataset reasoning.
  Example:
    "Create a histogram of sepal_length using matplotlib. Set figure size to 10 by 6. Use ASCII text only."

  If no chart is requested, set chartprompt to `False`.

Rules:
1. Never generate SQL or SQL-like language.
2. sql_intent must describe data logically, not programmatically.
3. chartprompt must only describe how to draw the chart, not what data to query.
4. Always include figure size instructions in chartprompt.
5. Output must be strictly valid JSON.

Schema for reference:
{schema}

User query:
{query}

Return ONLY valid JSON. No explanations.
"""


def parse_user_request(nlpquery: str):
    schema = getschema_agent()
    prompt = ChatPromptTemplate.from_template(PROMPT)
    chain = prompt | llm | JsonOutputParser()
    return chain.invoke({"schema": schema, "query": nlpquery})






In [63]:
respose=parse_user_request("Get all the products names and their quantities who its distribution using a plot")
respose

{'sql_intent': 'Return product names and quantity_available for all products.',
 'chartprompt': 'Create a bar chart of quantity_available with product_name on the x-axis using matplotlib. Set figure size to 10 by 6. Use ASCII text only.'}

In [64]:
sql_statement=respose.get("sql_intent")


In [56]:
# respose=parse_user_request("Plot the total revenue generated per month across the current year.")
sql_result=run_text_to_sql(sql_statement)

In [57]:
sql_result

{'success_status': True,
 'results': [{'product_name': 'Basmati Rice', 'quantity_available': 800},
  {'product_name': 'Whole Wheat', 'quantity_available': 600},
  {'product_name': 'Brown Sugar', 'quantity_available': 400},
  {'product_name': 'Green Tea', 'quantity_available': 200},
  {'product_name': 'Instant Coffee', 'quantity_available': 250},
  {'product_name': 'Black Tea', 'quantity_available': 180},
  {'product_name': 'Cashew Nuts', 'quantity_available': 120},
  {'product_name': 'Almonds', 'quantity_available': 150},
  {'product_name': 'Turmeric Powder', 'quantity_available': 300},
  {'product_name': 'Butter', 'quantity_available': 100}]}

In [65]:
import pandas as pd
from decimal import Decimal

# Build DataFrame properly
df = pd.DataFrame(sql_result["results"])

# Convert Decimals to float
for col in df.columns:
    if df[col].dtype == "object":
        if isinstance(df[col].iloc[0], Decimal):
            df[col] = df[col].astype(float)

# Normalize SQL auto-generated column names
rename_map = {}
for col in df.columns:
    clean = col.split("(")[0].replace(")", "").strip()
    rename_map[col] = clean

df.rename(columns=rename_map, inplace=True)

print(df)
print(df.columns)


      product_name  quantity_available
0     Basmati Rice                 800
1      Whole Wheat                 600
2      Brown Sugar                 400
3        Green Tea                 200
4   Instant Coffee                 250
5        Black Tea                 180
6      Cashew Nuts                 120
7          Almonds                 150
8  Turmeric Powder                 300
9           Butter                 100
Index(['product_name', 'quantity_available'], dtype='object')


In [66]:
from models.openaiapipoint import llm
import pandas as pd
from pandasai import SmartDataframe

# Load dataset
query=df

smartdataprompt=respose.get('chartprompt')
smartdataprompt

'Create a bar chart of quantity_available with product_name on the x-axis using matplotlib. Set figure size to 10 by 6. Use ASCII text only.'

In [67]:
sdf = SmartDataframe(df, config={"llm": llm})

result = sdf.chat(
   smartdataprompt
)

result

'c:/Users/ASUS/Desktop/langchain-gorq/Langraph/exports/charts/temp_chart.png'